In [1]:
from langchain.chat_models import ChatOllama
llm = ChatOllama(model="ollama_chat/llama3.2:3b", temperature=0)

C:\Users\basil\AppData\Local\Temp\ipykernel_13084\1981447662.py:2: LangChainDeprecationWarning: The class `ChatOllama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import ChatOllama``.
  llm = ChatOllama(model="ollama_chat/llama3.2:3b", temperature=0)


In [2]:
from crewai import Agent, Task, Crew


extractor_agent = Agent(
    role="Job-Relevant Skill Extractor",
    goal="Extract all skills and years of experience from the given CV TEXT.",
    backstory=(
        "You are an expert in NLP ,  resume parsing and job matching. "
        "You carefully analyze CVs and job descriptions to identify the most relevant skills that can be usefull to any job and the candidate's years of experience, "
        "ensuring that only skills applicable to the job requirements are included."
    ),
    llm = llm,
    verbose=True
)


checker_agent = Agent(
    role="Skill and Experience Validator",
    goal="Review the extracted skills and years of experience to ensure they match the actual CV content and are directly related to any job description and not generic or unrelated.",
    backstory=(
        "You are a specialist in candidate-job fit assessment. "
        "You validate that all extracted skills and experience are relevant to the job description, "
        "removing any unrelated or generic information and confirming the years of experience are accurately represented."
    ),
    llm = llm,
    verbose=False
)

In [5]:
def extract_and_check(cv_text):
    extract_task = Task(
        description=(
        "Extract EVERY skill, tool, framework, and technology explicitly mentioned in the following CV TEXT, "
        "including those mentioned in project descriptions, work experience, and technical skills sections. "
        "ONLY extract years of experience if it is explicitly stated in the text (such as 'X years of experience', 'since 2019', or similar). "
        "Do NOT infer years of experience from education dates, project dates, or any other indirect information. "
        "If years of experience is not explicitly mentioned, always use '0 years of experience'. "
        "List all skills, separated by spaces, after the years of experience. "
        "Format: '<years> years of experience skill1 skill2 skill3 ...'.\n"
        "Example: '2 years of experience python django restapi mysql'.\n"
        "**Do not include any explanation, thoughts, or reasoning. Output only the final answer string.**\n"
        f"CV:\n{cv_text}"
    ),
    expected_output="A single string with experience and relevant skills, e.g., '1 year of experience php laravel mysql restapi docker'.",
    agent=extractor_agent
    )

    check_task = Task(
        description=(
            "Step 2: Review the extracted string below from Step 1. Then, validate it against the full CV.\n\n"
            "Extracted Result:\n<<TASK_RESULT>>\n\n"
            "Full CV:\n"
            f"{cv_text}\n\n"
            "Ensure:\n"
            "- All skills are clearly present in the CV\n"
            "- No hallucinated, unrelated, or AI-related skills are included unless explicitly mentioned\n"
            "- The years of experience are realistically inferred from the text\n\n"
            "- No skills are missing or hallucinated\n"
            "- Do not include any thoughts, reasoning, or explanation. Output only the final answer string.\n"
            "Return the final string in the exact format:\n"
            "'<years> years of experience skill1 skill2 skill3 ...'"
        ),
        expected_output="A single string, e.g., '10 year of experience php laravel mysql restapi docker'.",
        agent=checker_agent,
        depends_on=extract_task
    )

    crew = Crew(
        agents=[extractor_agent, checker_agent],
        tasks=[extract_task, check_task],
        verbose=True
    )
    result = crew.kickoff()
    return str(result)

In [6]:
text = 'Ihab Helmy Mohamed Machine Learning Engineer ihab7lmy@gmail.com +201553421246 Mansoura, Egypt LinkedIn: linkedin.com/in/ihab-helmy Career Objective Machine Learning Engineer skilled in data science, NLP, and AI systems, with experience building end-to-end solutions using modern Python frameworks. Passionate about delivering impactful, scalable products and continuous learning. Training •National Telecommunication Institute (NTI) – July 2023 Education Bachelor of Science in Engineering Mansoura University, Department of Computer and Control Systems (2020 – 2025) Skills Technical Skills: Python, MySQL, Machine Learning, Deep Learning, NLP, Data Analysis, PyTorch, FastAPI, Web Scraping, NumPy, Pandas, Matplotlib, Seaborn, scikit-learn, LangChain, Hugging Face, Git & GitHub. Soft Skills: Problem Solving, Communication, Teamwork, Adaptability, Leadership, Ethics, Innovation. Projects •Chatbot: Developed an AI chatbot for a job hiring website using OpenAI API and FastAPI. The system answers user questions, helps freelancers write proposals, and assists employers with job post- ings. •Job Scraper: Built a robust web scraper to collect and preprocess 20,000+ job listings from platforms such as Indeed and Glassdoor, enabling data-driven analytics and ML projects. •Food Image Classification: Designed and trained a CNN for food image recognition, leveraging data augmentation to improve model performance. Languages •English (Professional working proficiency) •Arabic (Native) 1'
result = extract_and_check(text)
print(result)

╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: ed2153cb-c6fe-4bbe-9ba9-95a76fe4872d                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Job-Relevant Skill Extractor                                                                            │
│                                                                                                                 │
│  Task: Extract EVERY skill, tool, framework, and technology explicitly mentioned in the following CV TEXT,      │
│  including those mentioned in project descriptions, work experience, and technical skills sections. ONLY        │
│  extract years of experience if it is explicitly stated in the text (such as 'X years of experience', 'since    │
│  2019', or similar). Do NOT infer years of experience from education dates, project dates, or any other         │
│  indirect information. If years of experience is not explicitly mentioned, always use '0 years of experience'.  │
│  List all skills, separated by spaces, after the years of experience. Format: '<years> years of experience      │
│  skill1 skill2 skill3 ...'.                                                                                     │
│  Example: '2 years of experience python django restapi mysql'.                                                  │
│  **Do not include any explanation, thoughts, or reasoning. Output only the final answer string.**               │
│  CV:                                                                                                            │
│  Ihab Helmy Mohamed Machine Learning Engineer ihab7lmy@gmail.com +201553421246 Mansoura, Egypt LinkedIn:        │
│  linkedin.com/in/ihab-helmy Career Objective Machine Learning Engineer skilled in data science, NLP, and AI     │
│  systems, with experience building end-to-end solutions using modern Python frameworks. Passionate about        │
│  delivering impactful, scalable products and continuous learning. Training •National Telecommunication          │
│  Institute (NTI) – July 2023 Education Bachelor of Science in Engineering Mansoura University, Department of    │
│  Computer and Control Systems (2020 – 2025) Skills Technical Skills: Python, MySQL, Machine Learning, Deep      │
│  Learning, NLP, Data Analysis, PyTorch, FastAPI, Web Scraping, NumPy, Pandas, Matplotlib, Seaborn,              │
│  scikit-learn, LangChain, Hugging Face, Git & GitHub. Soft Skills: Problem Solving, Communication, Teamwork,    │
│  Adaptability, Leadership, Ethics, Innovation. Projects •Chatbot: Developed an AI chatbot for a job hiring      │
│  website using OpenAI API and FastAPI. The system answers user questions, helps freelancers write proposals,    │
│  and assists employers with job post- ings. •Job Scraper: Built a robust web scraper to collect and preprocess  │
│  20,000+ job listings from platforms such as Indeed and Glassdoor, enabling data-driven analytics and ML        │
│  projects. •Food Image Classification: Designed and trained a CNN for food image recognition, leveraging data   │
│  augmentation to improve model performance. Languages •English (Professional working proficiency) •Arabic       │
│  (Native) 1                                                                                                     │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Job-Relevant Skill Extractor                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  0 years of experience python machine learning deep learning nlp data analysis pytorch fastapi web scraping     │
│  numpy pandas matplotlib seaborn scikit-learn langchain hugging face git github english arabic                  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: c9e1b930-c226-4378-8838-fe0f2297b3b3                                                                     │
│  Agent: Job-Relevant Skill Extractor                                                                            │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Skill and Experience Validator                                                                          │
│                                                                                                                 │
│  Task: Step 2: Review the extracted string below from Step 1. Then, validate it against the full CV.            │
│                                                                                                                 │
│  Extracted Result:                                                                                              │
│  <<TASK_RESULT>>                                                                                                │
│                                                                                                                 │
│  Full CV:                                                                                                       │
│  Ihab Helmy Mohamed Machine Learning Engineer ihab7lmy@gmail.com +201553421246 Mansoura, Egypt LinkedIn:        │
│  linkedin.com/in/ihab-helmy Career Objective Machine Learning Engineer skilled in data science, NLP, and AI     │
│  systems, with experience building end-to-end solutions using modern Python frameworks. Passionate about        │
│  delivering impactful, scalable products and continuous learning. Training •National Telecommunication          │
│  Institute (NTI) – July 2023 Education Bachelor of Science in Engineering Mansoura University, Department of    │
│  Computer and Control Systems (2020 – 2025) Skills Technical Skills: Python, MySQL, Machine Learning, Deep      │
│  Learning, NLP, Data Analysis, PyTorch, FastAPI, Web Scraping, NumPy, Pandas, Matplotlib, Seaborn,              │
│  scikit-learn, LangChain, Hugging Face, Git & GitHub. Soft Skills: Problem Solving, Communication, Teamwork,    │
│  Adaptability, Leadership, Ethics, Innovation. Projects •Chatbot: Developed an AI chatbot for a job hiring      │
│  website using OpenAI API and FastAPI. The system answers user questions, helps freelancers write proposals,    │
│  and assists employers with job post- ings. •Job Scraper: Built a robust web scraper to collect and preprocess  │
│  20,000+ job listings from platforms such as Indeed and Glassdoor, enabling data-driven analytics and ML        │
│  projects. •Food Image Classification: Designed and trained a CNN for food image recognition, leveraging data   │
│  augmentation to improve model performance. Languages •English (Professional working proficiency) •Arabic       │
│  (Native) 1                                                                                                     │
│                                                                                                                 │
│  Ensure:                                                                                                        │
│  - All skills are clearly present in the CV                                                                     │
│  - No hallucinated, unrelated, or AI-related skills are included unless explicitly mentioned                    │
│  - The years of experience are realistically inferred from the text                                             │
│                                                                                                                 │
│  - No skills are missing or hallucinated                                                                        │
│  - Do not include any thoughts, reasoning, or explanation. Output only the final answer string.                 │
│  Return the final string in the exact format:                                                                   │
│  '<years> years of experience skill1 skill2 skill3 ...'

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Skill and Experience Validator                                                                          │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  0 years of experience Python Machine Learning Deep Learning NLP Data Analysis PyTorch FastAPI Web Scraping     │
│  NumPy Pandas Matplotlib Seaborn Scikit-learn LangChain Hugging Face Git GitHub English Arabic                  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: c792e613-db1a-4ed9-87c0-27835fcaa4da                                                                     │
│  Agent: Skill and Experience Validator                                                                          │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: ed2153cb-c6fe-4bbe-9ba9-95a76fe4872d                                                                       │
│  Tool Args:                                                                                                     │
│  Final Output: 0 years of experience Python Machine Learning Deep Learning NLP Data Analysis PyTorch FastAPI    │
│  Web Scraping NumPy Pandas Matplotlib Seaborn Scikit-learn LangChain Hugging Face Git GitHub English Arabic     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

0 years of experience Python Machine Learning Deep Learning NLP Data Analysis PyTorch FastAPI Web Scraping NumPy Pandas Matplotlib Seaborn Scikit-learn LangChain Hugging Face Git GitHub English Arabic


In [7]:
print(type(result))

<class 'str'>


In [22]:
text2 = "5 sequential years of experience on Data Analyst/ Business Analyst ALY AHMED 01091770006 alyahmed0022@gmail.com www.linkedin.com/in/aly -ahmed -11599532a https://github.com/AlyAhmedTS13/my_projects Skills • SQL (MySQL ) • Python (Pandas, NumPy, seaborn , MatPlotLib) • Web Scrapping /API • Machine Learning • Excel (VLookup, Conditional Formatting, Pivot Tables) • Microsoft Power BI Projects MY SQL WORLD LAYOFFS DATA CLE ANING AND EDA – Personal Project August 2024 • SQL Data Cleaning and Transformation : Cleaned and transformed a dataset of 2300+ rows with 9 columns using advanced SQL techniques such as window functions, STR to DATE conversion, subqueries, and column updates to enhance data accuracy and usability • Data Aggregation and Analysis : Conducted Exploratory Data Analysis (EDA) on the cleaned data, leveraging SQL's aggregate functions like MAX, MIN, and COUNT to extract valuable insights on global layoff trends. • Database Optimization and Feature Engineering : Streamlined the dataset by deleting irrelevant columns and creating new ones, optimizing the database for better analysis and reporting. POWER BI COMPANIE S AND THEIR REVE NUE DATA CLEANING AND VISUALZTION – Personal Project May 2024 ● Power BI Data Cleaning and Transformation : Managed and cleaned a dataset with over 10 columns and 5000 rows, applying conditional formatting and drill -down menus to improve data clarity and interactivity. ● DAX Function Usage for Insights : Utilized DAX functions like SUM and COUNT to perform data aggregation, enabling deeper analysis of combine rankings and producing actionable insights. ● Comprehensive Data Visualization : Created dynamic visualizations, including stacked bar charts and line charts, to effectively represent trends and comparisons, enhancing data -driven decision -making and made a compelling dashboard API CRYPTO CURRE NCY DATA SCRAPING USING PANDAS – Personal Project May 2024 ● Automated Data Scraping with BeautifulSoup : Scraped cryptocurrency data from a website using the BeautifulSoup library and automated the process with time and sleep functions to continuously update data over a specified time frame ● Data Handling and Cleaning with Pandas : Stored the scraped data in a Pandas DataFrame, performing minor cleaning operations to prepare the dataset for analysis and visualization. ● Data Visualization with Matplotlib : Used Matplotlib to create graphs that tracked changes in cryptocurrency values over time, providing a clear visual representation of market trends. EXCEL FULL PROJECT (DATA CLE ANING AND E DA) AND M AKING A DASHB ORAD – Personal Project July 2024 ● Data Cleaning and Transformation in Excel: Cleaned a dataset of over 1000 rows and 10+ columns using Excel functions like TRI M, nested IFs, and removed duplicates. Ensured data consistency by converting columns to appropriate data types. ● Data Analysis with Pivot Tables : Created pivot tables to effectively analyze and summarize the data, providing clear insights into biker buyer trends and behavior. ● Interactive Dashboard Creation : Developed an interactive Excel dashboard that visualized key insights, making the dataset more accessible and actionable for decision -making. Education MANSO URA UN IVERSITY (2020 –PRESENT) FACULTY O F E LECTRON ICS AN D COMM UNICATION DE PARTMENT"
result2 = extract_and_check(text2)
print(result2)

╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 304e7abb-3037-47dc-938c-81553152d32e                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Job-Relevant Skill Extractor                                                                            │
│                                                                                                                 │
│  Task: Extract EVERY skill, tool, framework, and technology explicitly mentioned in the following CV TEXT,      │
│  including those mentioned in project descriptions, work experience, and technical skills sections. ONLY        │
│  extract years of experience if it is explicitly stated in the text (such as 'X years of experience', 'since    │
│  2019', or similar). Do NOT infer years of experience from education dates, project dates, or any other         │
│  indirect information. If years of experience is not explicitly mentioned, always use '0 years of experience'.  │
│  List all skills, separated by spaces, after the years of experience. Format: '<years> years of experience      │
│  skill1 skill2 skill3 ...'.                                                                                     │
│  Example: '2 years of experience python django restapi mysql'.                                                  │
│  **Do not include any explanation, thoughts, or reasoning. Output only the final answer string.**               │
│  CV:                                                                                                            │
│  5 sequential years of experience on Data Analyst/ Business Analyst ALY AHMED 01091770006                       │
│  alyahmed0022@gmail.com www.linkedin.com/in/aly -ahmed -11599532a https://github.com/AlyAhmedTS13/my_projects   │
│  Skills • SQL (MySQL ) • Python (Pandas, NumPy, seaborn , MatPlotLib) • Web Scrapping /API • Machine Learning   │
│  • Excel (VLookup, Conditional Formatting, Pivot Tables) • Microsoft Power BI Projects MY SQL WORLD LAYOFFS     │
│  DATA CLE ANING AND EDA – Personal Project August 2024 • SQL Data Cleaning and Transformation : Cleaned and     │
│  transformed a dataset of 2300+ rows with 9 columns using advanced SQL techniques such as window functions,     │
│  STR to DATE conversion, subqueries, and column updates to enhance data accuracy and usability • Data           │
│  Aggregation and Analysis : Conducted Exploratory Data Analysis (EDA) on the cleaned data, leveraging SQL's     │
│  aggregate functions like MAX, MIN, and COUNT to extract valuable insights on global layoff trends. • Database  │
│  Optimization and Feature Engineering : Streamlined the dataset by deleting irrelevant columns and creating     │
│  new ones, optimizing the database for better analysis and reporting. POWER BI COMPANIE S AND THEIR REVE NUE    │
│  DATA CLEANING AND VISUALZTION – Personal Project May 2024 ● Power BI Data Cleaning and Transformation :        │
│  Managed and cleaned a dataset with over 10 columns and 5000 rows, applying conditional formatting and drill    │
│  -down menus to improve data clarity and interactivity. ● DAX Function Usage for Insights : Utilized DAX        │
│  functions like SUM and COUNT to perform data aggregation, enabling deeper analysis of combine rankings and     │
│  producing actionable insights. ● Comprehensive Data Visualization : Created dynamic visualizations, including  │
│  stacked bar charts and line charts, to effectively represent trends and comparisons, enhancing data -driven    │
│  decision -making and made a compelling dashboard API CRYPTO CURRE NCY DATA SCRAPING USING PANDAS – Personal    │
│  Project May 2024 ● Automated Data Scraping with BeautifulSoup : Scraped cryptocurrency data from a website     │
│  using the BeautifulSoup library and automated the process with time and sleep functions to continuously        │
│  update data over a specified time frame ● Data Handlin

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Job-Relevant Skill Extractor                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  5 years of experience sql python pandas seaborn matplotlib excel microsoft power bi machine learning mysql     │
│  web scrapping/api                                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 10252c33-c6e7-4c26-853b-f958706c484b                                                                     │
│  Agent: Job-Relevant Skill Extractor                                                                            │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Skill and Experience Validator                                                                          │
│                                                                                                                 │
│  Task: Step 2: Review the extracted string below from Step 1. Then, validate it against the full CV.            │
│                                                                                                                 │
│  Extracted Result:                                                                                              │
│  <<TASK_RESULT>>                                                                                                │
│                                                                                                                 │
│  Full CV:                                                                                                       │
│  5 sequential years of experience on Data Analyst/ Business Analyst ALY AHMED 01091770006                       │
│  alyahmed0022@gmail.com www.linkedin.com/in/aly -ahmed -11599532a https://github.com/AlyAhmedTS13/my_projects   │
│  Skills • SQL (MySQL ) • Python (Pandas, NumPy, seaborn , MatPlotLib) • Web Scrapping /API • Machine Learning   │
│  • Excel (VLookup, Conditional Formatting, Pivot Tables) • Microsoft Power BI Projects MY SQL WORLD LAYOFFS     │
│  DATA CLE ANING AND EDA – Personal Project August 2024 • SQL Data Cleaning and Transformation : Cleaned and     │
│  transformed a dataset of 2300+ rows with 9 columns using advanced SQL techniques such as window functions,     │
│  STR to DATE conversion, subqueries, and column updates to enhance data accuracy and usability • Data           │
│  Aggregation and Analysis : Conducted Exploratory Data Analysis (EDA) on the cleaned data, leveraging SQL's     │
│  aggregate functions like MAX, MIN, and COUNT to extract valuable insights on global layoff trends. • Database  │
│  Optimization and Feature Engineering : Streamlined the dataset by deleting irrelevant columns and creating     │
│  new ones, optimizing the database for better analysis and reporting. POWER BI COMPANIE S AND THEIR REVE NUE    │
│  DATA CLEANING AND VISUALZTION – Personal Project May 2024 ● Power BI Data Cleaning and Transformation :        │
│  Managed and cleaned a dataset with over 10 columns and 5000 rows, applying conditional formatting and drill    │
│  -down menus to improve data clarity and interactivity. ● DAX Function Usage for Insights : Utilized DAX        │
│  functions like SUM and COUNT to perform data aggregation, enabling deeper analysis of combine rankings and     │
│  producing actionable insights. ● Comprehensive Data Visualization : Created dynamic visualizations, including  │
│  stacked bar charts and line charts, to effectively represent trends and comparisons, enhancing data -driven    │
│  decision -making and made a compelling dashboard API CRYPTO CURRE NCY DATA SCRAPING USING PANDAS – Personal    │
│  Project May 2024 ● Automated Data Scraping with BeautifulSoup : Scraped cryptocurrency data from a website     │
│  using the BeautifulSoup library and automated the process with time and sleep functions to continuously        │
│  update data over a specified time frame ● Data Handling and Cleaning with Pandas : Stored the scraped data in  │
│  a Pandas DataFrame, performing minor cleaning operations to prepare the dataset for analysis and               │
│  visualization. ● Data Visualization with Matplotlib : Used Matplotlib to create graphs that tracked changes    │
│  in cryptocurrency values over time, providing a clear visual representation of market trends. EXCEL FULL       │
│  PROJECT (DATA CLE ANING AND E DA) AND M AKING A DASHB 

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Skill and Experience Validator                                                                          │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  5 years of experience SQL MySQL Python Pandas Seaborn Matplotlib Excel Microsoft Power BI Machine Learning     │
│  Web Scrapping/API                                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: a384dd87-2a08-448a-bdeb-2ad630ff6a52                                                                     │
│  Agent: Skill and Experience Validator                                                                          │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 304e7abb-3037-47dc-938c-81553152d32e                                                                       │
│  Tool Args:                                                                                                     │
│  Final Output: 5 years of experience SQL MySQL Python Pandas Seaborn Matplotlib Excel Microsoft Power BI        │
│  Machine Learning Web Scrapping/API                                                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

5 years of experience SQL MySQL Python Pandas Seaborn Matplotlib Excel Microsoft Power BI Machine Learning Web Scrapping/API
